# Ensemble Builder — Predictive AI Evaluation Challenge

This notebook turns the six Codabench submission bundles in `ensemble_bundles/` into a single weighted-ensemble Codabench submission.

**Pipeline**

1. **Environment + repo** — install deps, clone the project repo, check GPU.
2. **Bundles** — read the six pre-built submissions out of the repo, extract them, and report their identities.
3. **Validation slice** — download the public training data, build an item-cold-start val split, sample one stratified “official-like” round.
4. **Per-model predictions** — load each bundle sequentially (so we never have to fit five 4–8B encoders in VRAM at once), run `predict()` over the val round, save the probability vectors and a copy of the same `labeled` list every model saw.
5. **Diversity analysis** — pairwise Pearson, mean absolute disagreement, KL, Yule’s Q, double-fault, per-model log-loss / Brier.
6. **Weight optimisation** — pick a subset (the recommended path is to drop highly-correlated under-performers) and fit weights in *probability* and *logit* space, plus an unconstrained-logit stacker. We print the resulting val log-loss / Brier alongside the uniform baseline.
7. **Build ensemble bundle** — produce a self-contained `ensemble_submission.zip` you can upload to Codabench directly.
8. **Smoke test** — reload the produced bundle as if we were Codabench and run `predict()` on a handful of rows.
9. **Download** — `files.download()` the final zip.

**Recommended runtime**: Colab Pro A100 (40 GB) or A100 High-RAM (80 GB). The diversity sweep loads only one encoder at a time, so 40 GB is enough; the *final ensemble*, however, holds every selected submodel co-resident, so picking >3 large encoders requires 80 GB or a smaller subset.

**Cost reality-check before you start**

* Encoder downloads alone are 30–50 GB depending on which submodels you pick (Codabench will fetch them again from HF at evaluation; here we cache them via `huggingface_hub`).
* Each per-model val pass takes 5–15 minutes for the 4B/8B encoders, mostly for item-embedding warm-up.
* You can shrink the val slice with `VAL_N` in section 3 below — 1–2k rows is plenty for diversity work; the *ensemble weights* are 3–6 numbers and won’t overfit to a small val.


## 1. Environment + GPU info

We just print sizes — you can switch GPU type from the Runtime menu before running this cell.

In [ ]:
import os, sys, json, time, shutil, subprocess
from pathlib import Path

print('Python:', sys.version.split()[0])
try:
    import torch
    print('torch:', torch.__version__, 'cuda:', torch.version.cuda, 'cuda_available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        free, total = torch.cuda.mem_get_info(0)
        print(f'GPU memory: free={free/1e9:.2f}GB total={total/1e9:.2f}GB')
except ImportError:
    print('torch is not installed yet — will install in next cell.')
subprocess.run(['nvidia-smi'], check=False)

## 2. Install dependencies

Colab ships `torch`, `pandas`, `numpy`, `scikit-learn`, `pyarrow`. We add:
* `transformers`, `huggingface_hub`, `safetensors`, `accelerate` — needed by every encoder bundle.
* `sentencepiece`, `protobuf` — needed by SFR-Embedding-Mistral and llama-embed-nemotron tokenizers.
* `langdetect` — used in the pool features path of every submodel.
* `einops` — needed by some encoder implementations.

Colab’s torch is pinned; we explicitly `--upgrade` only the safe packages.

In [ ]:
%pip install -q --upgrade transformers huggingface_hub safetensors accelerate sentencepiece protobuf langdetect einops pyarrow

## 3. Clone the repo

The repo contains:
* `ensemble_bundles/*.zip` — the six prebuilt submissions.
* `src/ensemble_helpers.py` — the analysis + weight-fitting utilities used below.
* `scripts/build_ensemble_submission.py` — the bundle‐packing CLI.
* `validation_harness/` — the official-like round simulator.
* `src/data.py` — the HF parquet downloader / joiner.

If you need to point at a fork, edit `REPO_URL`.

In [ ]:
REPO_URL = 'https://github.com/bwathomas/prediction-competition-321M.git'
REPO_DIR = Path('/content/Prediction-Competition-321M').resolve()
if REPO_DIR.exists():
    print(f'{REPO_DIR} already exists, pulling latest …')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

for sub in ['ensemble_bundles', 'src', 'scripts', 'validation_harness']:
    p = REPO_DIR / sub
    print(f'  {sub:25s} {"OK" if p.exists() else "MISSING"}')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if str(REPO_DIR / 'validation_harness') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'validation_harness'))
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

## 4. Catalog the bundles

Each bundle ships its own `runtime_meta.json` describing the encoder, pooling and max length. We summarise them so you can see what you’re ensembling.

In [ ]:
import zipfile
import pandas as pd

BUNDLE_DIR = REPO_DIR / 'ensemble_bundles'
WORK = Path('/content/ensemble_workspace')
WORK.mkdir(parents=True, exist_ok=True)
EXTRACTED_DIR = WORK / 'extracted'
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

from src.ensemble_helpers import extract_bundle

bundles_info = []
for zip_path in sorted(BUNDLE_DIR.glob('*.zip')):
    name = zip_path.stem
    out_dir = EXTRACTED_DIR / name
    extract_bundle(zip_path, out_dir)
    meta_candidates = [
        out_dir / 'artifacts' / 'runtime_meta.json',
        out_dir / 'runtime_meta.json',
    ]
    meta = {}
    for mp in meta_candidates:
        if mp.is_file():
            try:
                meta = json.loads(mp.read_text(encoding='utf-8'))
            except Exception as e:
                print(f'  could not read {mp}: {e}')
            break
    bundles_info.append({
        'name': name,
        'zip_mb': round(zip_path.stat().st_size / 1e6, 2),
        'encoder_model_id': meta.get('encoder_model_id') or '<no encoder / LFM>',
        'pooling': meta.get('pooling', ''),
        'max_length': meta.get('max_length', ''),
        'arch': meta.get('runtime_architecture', meta.get('model_name', '')),
        'best_val_log_loss': meta.get('best_val_log_loss'),
    })
bundles_df = pd.DataFrame(bundles_info)
bundles_df

## 5. Download the public dataset and build a val slice

We pull the full parquet collection from `aims-foundations/measurement-db`, then construct an item-cold-start split (so val items never appear in train) and assign 15 data categories the same way the platform does. The diversity / weight steps later need only ~5–10k rows, so we sub-sample the val set down to `VAL_N` rows before running any encoders.

If you’ve already run this cell once, the parquets are cached under `/content/measurement_db/`.

**Optional**: paste your HuggingFace token below if your IP is rate-limited. The dataset is public, so a token is *not* required.

In [ ]:
HF_TOKEN = None
VAL_N = 2500
MAX_ROWS_PER_BENCHMARK = 20000
VAL_FRACTION = 0.05
RANDOM_SEED = 7

from src.data import (
    add_stable_keys,
    binarize_labels,
    download_measurement_db,
    load_joined_responses,
)
from harness.data_loader import add_data_category
from harness.splits import (
    add_item_split_key,
    add_item_variant_id,
    make_item_cold_start_split,
)

DATA_DIR = Path('/content/measurement_db')
DATA_DIR.mkdir(parents=True, exist_ok=True)
download_measurement_db(local_dir=DATA_DIR, token=HF_TOKEN)

df = load_joined_responses(DATA_DIR, max_rows_per_benchmark=MAX_ROWS_PER_BENCHMARK)
print(f'joined rows: {len(df):,}')
df = add_stable_keys(df)
df = binarize_labels(df, keep_soft=False)
df = add_item_variant_id(df)
df = add_item_split_key(df)
df = add_data_category(df, mode='random', n_categories=15, seed=0)

train_df, val_df, val_unseen_df, report = make_item_cold_start_split(
    df,
    val_fraction=VAL_FRACTION,
    seed=RANDOM_SEED,
    variant_col='item_split_key',
)
print(f'train: {len(train_df):,}  val: {len(val_df):,}  val_unseen_subjects: {len(val_unseen_df):,}')
print(f'val benchmarks: {val_df["benchmark"].nunique()}  '
      f'val categories: {val_df["data_category"].nunique()}')

rng = pd.Series(val_df.index).sample(min(VAL_N, len(val_df)), random_state=RANDOM_SEED)
val_slice = val_df.loc[rng.values].reset_index(drop=True)
print(f'sub-sampled val_slice: {len(val_slice):,} rows')

val_slice.to_parquet(WORK / 'val_slice.parquet', index=False)

### Build a `labeled` list using random selection

Every submodel’s `predict()` takes a `labeled` argument: a list of dicts of the form `{benchmark, condition, subject_content, item_content, label}` corresponding to the rows the platform reveals (top-K per data category). We pre-build this list ONCE and pass the *same* list to every model, so the diversity numbers compare *like with like*.

We use **random selection** here (not any model’s `acquisition_function`) for two reasons:
1. It avoids accidentally biasing the comparison toward whichever model’s acquisition policy we use.
2. It removes the need to run an extra encoder pass *before* the actual prediction loop.

If you want to mimic the production behaviour more faithfully later, see section 9 (it shows how to drive each model’s native `labeling.acquisition_function` for a round, then re-run predict).

In [ ]:
import numpy as np
from src.ensemble_helpers import (
    df_to_inputs,
    df_to_labeled,
    select_topk_per_category,
)

K_PER_CATEGORY = 5
rng = np.random.default_rng(RANDOM_SEED)
rand_scores = rng.random(len(val_slice))
labeled_idx = select_topk_per_category(
    rand_scores,
    val_slice['data_category'].astype(str).to_numpy(),
    k_per_category=K_PER_CATEGORY,
    seed=RANDOM_SEED,
)
labeled_df = val_slice.loc[labeled_idx].reset_index(drop=True)
predict_df = val_slice.drop(index=labeled_idx).reset_index(drop=True)
print(f'labeled rows shown to predict(): {len(labeled_df)}')
print(f'predict-target rows: {len(predict_df)}')

labeled_list = df_to_labeled(labeled_df)
predict_inputs = df_to_inputs(predict_df)
predict_labels = predict_df['label'].astype(float).to_numpy()
print('label mean (predict-set):', predict_labels.mean().round(4))

## 6. Run every model sequentially

We load one bundle at a time, run `predict()` over `predict_inputs`, store the resulting probability vector, then drop the bundle’s state from `sys.modules` and `torch.cuda.empty_cache()` before the next.

On first run each submodel will trigger an encoder download from HuggingFace. Expect 5–15 minutes per model on Colab Pro A100; total round-trip 30–70 minutes for all six.

Predictions are persisted to `/content/ensemble_workspace/preds/<model>.npy` so you can re-run the diversity analysis without re-encoding.

In [ ]:
from src.ensemble_helpers import load_submodel, unload_submodel, run_predict_loop

PRED_DIR = WORK / 'preds'
PRED_DIR.mkdir(parents=True, exist_ok=True)

MODELS_TO_RUN = sorted(p.name for p in EXTRACTED_DIR.iterdir() if (p / 'model.py').exists())
print('will run:', MODELS_TO_RUN)

preds_by_model = {}
stats_by_model = {}
for name in MODELS_TO_RUN:
    out_path = PRED_DIR / f'{name}.npy'
    stat_path = PRED_DIR / f'{name}.stats.json'
    if out_path.exists() and stat_path.exists():
        print(f'[cached] {name}')
        preds_by_model[name] = np.load(out_path)
        stats_by_model[name] = json.loads(stat_path.read_text())
        continue
    print(f'\n=== {name} ===')
    sub_dir = EXTRACTED_DIR / name
    mod = load_submodel(sub_dir, name=f'_submod_{name}')
    try:
        preds, stats = run_predict_loop(mod, predict_inputs, labeled_list, label=name)
        preds_by_model[name] = preds
        stats_by_model[name] = stats
        np.save(out_path, preds)
        stat_path.write_text(json.dumps(stats, indent=2))
    finally:
        unload_submodel(mod)
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
print('\nfinished; cached predictions at', PRED_DIR)

### Sanity-check the predictions

We check that each model produced a real prediction (not the 0.5 fallback) on almost every row, and that we got finite probabilities. Models that fail here should be excluded from the ensemble.

In [ ]:
from src.ensemble_helpers import log_loss_vec, brier_vec

sanity_rows = []
for name, preds in preds_by_model.items():
    s = stats_by_model.get(name, {})
    finite = np.isfinite(preds)
    sanity_rows.append({
        'model': name,
        'n_rows': int(finite.sum()),
        'n_nan': int((~finite).sum()),
        'n_default_05': int(s.get('n_default_05', 0)),
        'seconds': round(s.get('seconds', 0.0), 1),
        'mean_p': float(preds[finite].mean()) if finite.any() else float('nan'),
        'log_loss': log_loss_vec(preds, predict_labels),
        'brier': brier_vec(preds, predict_labels),
    })
sanity_df = pd.DataFrame(sanity_rows).sort_values('log_loss')
sanity_df

## 7. Diversity analysis

Pairwise statistics over the predicted probabilities. Lower values of any metric (except Pearson) indicate *more* diversity between the two models.

*Heuristic for ensembling*: keep the best individual model and add the model that’s **least correlated** with it (highest mean-abs-diff, lowest Pearson). Repeat until adding a model no longer improves val log-loss in section 8.

In [ ]:
from src.ensemble_helpers import compute_diversity_metrics

report = compute_diversity_metrics(preds_by_model, predict_labels)
summary = report.diversity_summary()
summary

In [ ]:
import matplotlib.pyplot as plt

def _plot_heatmap(df, title, fmt='{:.3f}'):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(df.values, cmap='viridis')
    ax.set_xticks(range(len(df.columns)))
    ax.set_yticks(range(len(df.index)))
    ax.set_xticklabels(df.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(df.index, fontsize=8)
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            v = df.values[i, j]
            if np.isfinite(v):
                ax.text(j, i, fmt.format(v), ha='center', va='center',
                        color='white' if v < df.values.mean() else 'black', fontsize=7)
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

_plot_heatmap(report.pearson, 'Pairwise Pearson correlation (higher = less diverse)')
_plot_heatmap(report.mean_abs_diff, 'Pairwise mean |p_a - p_b| (higher = more diverse)')
_plot_heatmap(report.kl_div, 'Symmetrised Bernoulli KL (higher = more diverse)')
_plot_heatmap(report.q_statistic, 'Yule\'s Q on correctness (1 = same errors, -1 = anti)')
_plot_heatmap(report.double_fault, 'Double-fault P(both wrong) (lower = more diverse on errors)')

## 8. Pick a subset and fit ensemble weights

Edit `SUBSET` below to choose which submodels to combine. The cell prints the val log-loss for several weighting schemes:

* **uniform_prob** — mean of probabilities. Robust baseline.
* **uniform_logit** — mean of logits. Often slightly better than `uniform_prob` for well-calibrated models.
* **simplex_prob** — non-negative weights summing to 1, fitted in probability space.
* **simplex_logit** — same, fitted in logit space.
* **unconstrained_logit** — stacking via logistic regression on per-model logits (intercept + arbitrary signs). The most flexible; the most likely to overfit if `VAL_N` is very small.

Pick the lowest `log_loss` (with a sanity check against `brier`) and proceed.

In [ ]:
from src.ensemble_helpers import fit_optimal_weights

SUBSET = list(preds_by_model.keys())
print('subset:', SUBSET)
preds_subset = {n: preds_by_model[n] for n in SUBSET}

fits = {}
for method in ['uniform_prob', 'uniform_logit', 'simplex_prob', 'simplex_logit', 'unconstrained_logit']:
    try:
        fit = fit_optimal_weights(preds_subset, predict_labels, method=method)
        fits[method] = fit
    except Exception as e:
        print(f'  {method}: FAILED {e}')

rows = []
for method, fit in fits.items():
    rows.append({
        'method': method,
        'space': fit.space,
        'log_loss': fit.log_loss,
        'brier': fit.brier,
        'weights': np.round(fit.weights, 4).tolist(),
        'notes': fit.notes,
    })
fit_summary = pd.DataFrame(rows).sort_values('log_loss').reset_index(drop=True)
fit_summary

In [ ]:
BEST_METHOD = fit_summary.iloc[0]['method']
BEST_FIT = fits[BEST_METHOD]
print(f'best method: {BEST_METHOD}  log_loss={BEST_FIT.log_loss:.5f}')
for n, w in zip(BEST_FIT.model_names, BEST_FIT.weights):
    print(f'  {n:35s} w={w: .5f}')

best_individual = float(min(report.per_model_loss.dropna()))
best_individual_name = report.per_model_loss.dropna().idxmin()
print(f'\nbest single model: {best_individual_name} (log_loss={best_individual:.5f})')
print(f'ensemble vs best single: {BEST_FIT.log_loss - best_individual:+.5f} nats')

## 9. (Optional) Replay with each model's native `acquisition_function`

Section 6 used *random* labels for the K·categories rows. The platform actually selects them via your `labeling.acquisition_function`. Replaying with each model’s own scorer is significantly more expensive (an extra encoder pass over the full val slice) but is the most faithful simulation of leaderboard behaviour.

Skip this section if you trust the random-label diversity numbers — the ensemble weights are nearly always insensitive to whose `acquisition_function` selected the 75 labels.

Below we show the code for one model; copy it per model you want to re-evaluate.

In [ ]:
# from src.ensemble_helpers import load_submodel, unload_submodel, run_acquisition_pass
# import importlib.util
# name = 'dropout03_fwls_combined'  # edit to taste
# sub_dir = EXTRACTED_DIR / name
# mod = load_submodel(sub_dir, name=f'_acq_{name}')
# # load labeling.py from the SAME directory
# spec = importlib.util.spec_from_file_location(f'_lbl_{name}', sub_dir / 'labeling.py')
# lbl_mod = importlib.util.module_from_spec(spec)
# sys.modules[spec.name] = lbl_mod
# # Inject the loaded model under the name `model` so labeling.py’s `from model import ...` works
# sys.modules['model'] = mod
# spec.loader.exec_module(lbl_mod)
# all_inputs = df_to_inputs(val_slice)
# scores, err = run_acquisition_pass(lbl_mod, all_inputs)
# print('err:', err)
# new_idx = select_topk_per_category(scores, val_slice['data_category'].astype(str).to_numpy(), k_per_category=5, seed=RANDOM_SEED)
# new_labeled = df_to_labeled(val_slice.iloc[new_idx])
# new_predict = val_slice.drop(index=new_idx).reset_index(drop=True)
# new_inputs = df_to_inputs(new_predict)
# new_preds, _ = run_predict_loop(mod, new_inputs, new_labeled, label=f'{name}_native')
# unload_submodel(mod)

## 10. Build the Codabench ensemble bundle

We package the chosen subset with the fitted weights into a single `ensemble_submission.zip`.

**Bundle layout** (auto-built by `scripts/build_ensemble_submission.py`):
```
ensemble_submission.zip/
  model.py                 # imports each submodel, weighted-combines predict()
  labeling.py              # imports SUBMODELS from model.py, drives each enqueue
  models.txt               # union of every submodel's HF repos (deduped)
  ensemble_meta.json       # weights, combine_space, intercept, names
  submodels/<name>/        # full copy of each chosen bundle
    model.py
    labeling.py
    artifacts/…
```

**Codabench platform expectations** — read these before submitting:
* The platform allows at most **5** entries in `models.txt`. If your subset has 5 LLM encoders, the LFM-only bundle (which has *no* HF repo) is free; if you pick 5 LLM + LFM you’re still at 5. If you somehow end up >5, the script will *warn* and proceed; Codabench may reject.
* The platform routes by the *largest declared model*. All five LLM encoders here are 4–8 B params, so routing lands at the 8 B tier (L4 at 24 GB) unless you also declare a placeholder >8 B. **An ensemble of ≥3 large encoders will NOT fit in L4 VRAM**; you must either (a) shrink to 1–2 large encoders + the LFM, or (b) declare a dummy 20–70 B param model to force A100/A100-4 routing.
* The total tier timeout is 30–60 minutes. With 5 encoders each doing item embedding for 256 k acquisition calls, expect to budget aggressively — the streamed-flush queues per submodel each fire at most one batch per acquisition call, and 5 queues = 5× the work.

In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR / 'scripts'))
from build_ensemble_submission import build_ensemble

bundle_paths = [BUNDLE_DIR / f'{n}.zip' for n in BEST_FIT.model_names]
for p in bundle_paths:
    assert p.exists(), f'missing {p}'

combine_space = BEST_FIT.space  # 'prob' or 'logit'
intercept = 0.0
if BEST_FIT.method == 'unconstrained_logit':
    intercept = float(BEST_FIT.notes.split('=')[-1])

ENSEMBLE_OUT = WORK / 'ensemble_submission.zip'
build_ensemble(
    bundles=[str(p) for p in bundle_paths],
    weights=BEST_FIT.weights.tolist(),
    submodel_names=BEST_FIT.model_names,
    combine_space=combine_space,
    intercept=intercept,
    default_prob=0.5,
    out_zip=ENSEMBLE_OUT,
    extra_notes={
        'fit_method': BEST_FIT.method,
        'val_log_loss': BEST_FIT.log_loss,
        'val_brier': BEST_FIT.brier,
        'val_n_rows': BEST_FIT.n_rows,
        'random_seed': RANDOM_SEED,
        'val_n_input': VAL_N,
    },
)
print(f'wrote {ENSEMBLE_OUT}  ({ENSEMBLE_OUT.stat().st_size/1e6:.2f} MB)')

## 11. Smoke-test the produced bundle

We extract the zip into a fresh dir, importlib-load `model.py`, and call `predict()` on a handful of held-out rows. If this raises or returns 0.5 on every row, **do not upload to Codabench**.

Note that this is more strict than a normal submission test because we’re holding all submodels co-resident in VRAM — if it crashes here for OOM, the same will happen on Codabench unless you bumped to a higher tier.

In [ ]:
import importlib.util

SMOKE_DIR = WORK / 'smoke_test'
if SMOKE_DIR.exists():
    shutil.rmtree(SMOKE_DIR)
SMOKE_DIR.mkdir(parents=True)
with zipfile.ZipFile(ENSEMBLE_OUT) as zf:
    zf.extractall(SMOKE_DIR)
print('extracted to', SMOKE_DIR)

if str(SMOKE_DIR) not in sys.path:
    sys.path.insert(0, str(SMOKE_DIR))
# Drop any previously-loaded `model` / `labeling` modules so we get a clean import
for mod_name in list(sys.modules):
    if mod_name in ('model', 'labeling') or mod_name.startswith('_ensemble_submodel_'):
        sys.modules.pop(mod_name, None)

spec = importlib.util.spec_from_file_location('model', SMOKE_DIR / 'model.py')
ensemble_mod = importlib.util.module_from_spec(spec)
sys.modules['model'] = ensemble_mod
spec.loader.exec_module(ensemble_mod)
print('ensemble loaded with submodels:', ensemble_mod.SUBMODEL_NAMES_LOADED)
print('load errors:', ensemble_mod._SUBMODEL_LOAD_ERRORS)

smoke_inputs = predict_inputs[:5]
for i, inp in enumerate(smoke_inputs):
    p = ensemble_mod.predict(inp, labeled_list)
    print(f'  smoke {i}: p={p:.5f}')
    assert 1e-4 < p < 1.0 - 1e-4, f'bad probability: {p}'
print('SMOKE TEST OK')

## 12. Run one ensemble round end-to-end (recommended sanity check)

We run the full official-like round simulator against the produced ensemble bundle. This is slow (every acquisition call ENQUEUES into every submodel’s queue and every predict call calls every submodel’s `predict()`), but it’s the closest thing we have to the actual Codabench run.

Expect 10–40 minutes depending on the subset size and the encoder you picked.

In [ ]:
SKIP_FULL_ROUND = True  # flip to False if you really want to wait 10-40 minutes
if not SKIP_FULL_ROUND:
    # We bypass the harness's Submission wrapper because it loads model.py
    # and labeling.py under unique synthetic names; the ensemble's
    # labeling.py does `from model import SUBMODELS`, which requires the
    # bare name `model` to be in sys.modules. So we load manually in the
    # right order.
    from harness.rounds import run_official_like_round

    for mod_name in list(sys.modules):
        if mod_name in ('model', 'labeling') or mod_name.startswith('_ensemble_submodel_'):
            sys.modules.pop(mod_name, None)

    spec_m = importlib.util.spec_from_file_location('model', SMOKE_DIR / 'model.py')
    ensemble_mod = importlib.util.module_from_spec(spec_m)
    sys.modules['model'] = ensemble_mod
    spec_m.loader.exec_module(ensemble_mod)

    spec_l = importlib.util.spec_from_file_location('labeling', SMOKE_DIR / 'labeling.py')
    labeling_mod = importlib.util.module_from_spec(spec_l)
    sys.modules['labeling'] = labeling_mod
    spec_l.loader.exec_module(labeling_mod)

    round_result = run_official_like_round(
        train_df=train_df,
        val_df=val_slice,
        model_module=ensemble_mod,
        labeling_module=labeling_mod,
        N=min(1500, len(val_slice)),
        K=K_PER_CATEGORY,
        seed=RANDOM_SEED,
    )
    scored = round_result.candidates.dropna(subset=['_pred'])
    log_loss = log_loss_vec(scored['_pred'].to_numpy(), scored['label'].to_numpy())
    brier = brier_vec(scored['_pred'].to_numpy(), scored['label'].to_numpy())
    print(f'official-like round | log_loss={log_loss:.5f}  brier={brier:.5f}  '
          f'n_pred={len(scored)} fallback={round_result.fallback_reason}')
else:
    print('Skipping full official-like round (SKIP_FULL_ROUND=True). '
          'Set to False to validate the labeling path end-to-end.')

## 13. Download the ensemble zip

Run this cell to pull the zip onto your local machine, then submit it to Codabench via the web UI.

In [ ]:
try:
    from google.colab import files
    files.download(str(ENSEMBLE_OUT))
except Exception:
    print(f'Not running in Colab. Output zip is at: {ENSEMBLE_OUT}')

## 14. (Optional) Push the ensemble back to git

If you want the produced ensemble checkpoint tracked in the repo, run this cell. We push to `ensemble_outputs/` which is *not* gitignored (unlike `submission/`). Be aware that a 50–200 MB binary in git history is fine for one or two iterations but will bloat the repo if you iterate dozens of times.

In [ ]:
PUSH_TO_GIT = False  # flip to True to actually push
if PUSH_TO_GIT:
    dest = REPO_DIR / 'ensemble_outputs' / f'ensemble_{int(time.time())}.zip'
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ENSEMBLE_OUT, dest)
    subprocess.run(['git', '-C', str(REPO_DIR), 'add', str(dest.relative_to(REPO_DIR))], check=True)
    subprocess.run(
        ['git', '-C', str(REPO_DIR), '-c', 'user.email=colab@local', '-c', 'user.name=Colab',
         'commit', '-m', f'ensemble: weights={BEST_FIT.weights.tolist()} log_loss={BEST_FIT.log_loss:.5f}'],
        check=False,
    )
    print('committed; push manually with `git push` when you have credentials configured.')
else:
    print('PUSH_TO_GIT is False; skipping.')

---

## Appendix — Gotchas + Recommendations

1. **VRAM budget**: at runtime the ensemble holds **every** chosen submodel co-resident. With 5 LLM encoders that’s ~30–60 GB of bf16 weights. If you pick all 5, you must declare a placeholder ≥20B param model in `models.txt` so Codabench routes to A100-4 or higher. The script aggregates *every* submodel’s `models.txt`; if the union exceeds 5 entries it will warn but still proceed.
2. **Time budget per predict()**: each submodel’s `predict()` runs a `_flush_pending_batches()` first. Across 5 submodels that’s 5 flushes of up to one residual batch each — worst case ~5×1.5 = 7.5 s of compute. The platform’s per-call timeout is 10 s. Stay below 5 submodels with the streamed-flush architecture.
3. **Encoder warm-up math**: each submodel pre-encodes its unique items during acquisition. With ~5000 unique items in the val pool and bs=8, that’s ~625 batches per submodel. At ~1 s/batch on A100 that’s ~10 min per submodel — cumulative ~50 min for a 5-model ensemble. The platform timeout is 30 min (L4) or 60 min (A100). Aim for ≤3 large encoders OR pre-declare a higher tier.
4. **Calibrator scope**: each submodel’s `_Calibrator` fits on the *same* `labeled` list, so calibration ends up correlated across submodels. This is fine — a single round’s 75 labels don’t carry enough information to wildly diverge the per-bc intercepts. The diversity that matters comes from the encoder + IRT head, which is preserved.
5. **Hugging Face download time**: first run will download ~30–60 GB of encoder weights. Use the HF cache (`/root/.cache/huggingface/`) and consider mounting Google Drive so the cache survives session resets.
6. **Validation set leakage**: every LLM submodel was trained on item-cold-start splits of the public training data, but the *seed* for that split may differ from `RANDOM_SEED` here. Items in your val slice MAY have been seen by some submodels at training time. This biases the absolute log-loss numbers low, but the *relative* diversity numbers remain trustworthy because all submodels see the same val slice.
7. **Picking models**: as a rule of thumb, keep the top-2 individual log-loss models plus the model that has the lowest mean correlation with them. The unconstrained-logit stacker tends to win on the val set but is the most likely to overfit if `VAL_N` is small (< 1000). When in doubt, ship `simplex_logit` — it never assigns negative weights and is robust to one bad submodel.
